# Notebook 2 — Silver → Gold
**Tech Challenge Fase 3 · Big Data to Analytics · POSTECH**

Geração das tabelas analíticas (camada Gold) a partir dos dados consolidados da camada Silver.

**Tabelas geradas:**
- `diversidade_genero` — distribuição por gênero e ano
- `distribuicao_regional` — distribuição por região e ano
- `distribuicao_uf` — distribuição por estado e ano
- `cor_raca` — distribuição por cor/raça/etnia e ano
- `perfil_profissional` — colunas P2 (cargo, senioridade, salário, modelo de trabalho)
- `tecnologias` — colunas P4 (linguagens, cloud, BI, IA)

## 1. Inicialização

In [ ]:
import sys
import re
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F

args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

BUCKET = 's3://tc-fase3-state-of-data-wl'
SILVER = f'{BUCKET}/silver/state_of_data_consolidado/'
GOLD   = f'{BUCKET}/gold'
print('Contexto inicializado!')

## 2. Leitura da camada Silver e limpeza de colunas

In [ ]:
def limpar(nome):
    nome = re.sub(r'[^a-zA-Z0-9_]', '_', nome)
    nome = re.sub(r'_+', '_', nome)
    return nome.strip('_')

df = spark.read.parquet(SILVER)
df = df.toDF(*[limpar(c) for c in df.columns])

# Checkpoint para quebrar o lineage
spark.sparkContext.setCheckpointDir(f'{BUCKET}/checkpoints/')
df = df.checkpoint()

print(f'Total de registros: {df.count()}')
print(f'Total de colunas: {len(df.columns)}')
print(f'Colunas P2 disponíveis: {[c for c in df.columns if c.startswith("P2")][:5]}')
print(f'Colunas P4 disponíveis: {[c for c in df.columns if c.startswith("P4")][:5]}')

## 3. Gold — Diversidade de Gênero

In [ ]:
df.groupBy('ano_pesquisa', 'P1_b__Genero') \
  .agg(F.count('*').alias('total')) \
  .write.mode('overwrite') \
  .parquet(f'{GOLD}/diversidade_genero/')
print('Gold OK: diversidade_genero')

## 4. Gold — Distribuição Regional e por UF

In [ ]:
df.groupBy('ano_pesquisa', 'P1_e_b__Regiao_onde_mora') \
  .agg(F.count('*').alias('total')) \
  .write.mode('overwrite') \
  .parquet(f'{GOLD}/distribuicao_regional/')
print('Gold OK: distribuicao_regional')

df.groupBy('ano_pesquisa', 'P1_e_a__uf_onde_mora') \
  .agg(F.count('*').alias('total')) \
  .write.mode('overwrite') \
  .parquet(f'{GOLD}/distribuicao_uf/')
print('Gold OK: distribuicao_uf')

## 5. Gold — Cor / Raça / Etnia

In [ ]:
col_raca = [c for c in df.columns if 'raca' in c.lower() or 'etnia' in c.lower()]
print(f'Coluna raça encontrada: {col_raca}')

if col_raca:
    df.groupBy('ano_pesquisa', col_raca[0]) \
      .agg(F.count('*').alias('total')) \
      .write.mode('overwrite') \
      .parquet(f'{GOLD}/cor_raca/')
    print('Gold OK: cor_raca')

## 6. Gold — Perfil Profissional (colunas P2)

Contém: cargo, senioridade, faixa salarial, modelo de trabalho, satisfação, etc.

In [ ]:
colunas_p2 = [c for c in df.columns if c.startswith('P2')]
print(f'Colunas P2 encontradas: {len(colunas_p2)}')

df.select(['ano_pesquisa'] + colunas_p2) \
  .write.mode('overwrite') \
  .parquet(f'{GOLD}/perfil_profissional/')
print('Gold OK: perfil_profissional')

## 7. Gold — Tecnologias (colunas P4)

Contém: linguagens, fontes de dados, cloud, BI, adoção de IA generativa.

In [ ]:
colunas_p4 = [c for c in df.columns if c.startswith('P4')]
print(f'Colunas P4 encontradas: {len(colunas_p4)}')

if colunas_p4:
    df.select(['ano_pesquisa'] + colunas_p4) \
      .write.mode('overwrite') \
      .parquet(f'{GOLD}/tecnologias/')
    print('Gold OK: tecnologias')

print('=== Todas as Gold layers concluídas! ===')
job.commit()